In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, models, transforms

import numpy as np
import matplotlib.pyplot as plt
import os
import time
from PIL import Image

# --- Proje Sabitleri ---
DATA_DIR = "../../data/prepared_data_aug/" ,

TRAIN_DIR = os.path.join(DATA_DIR, "train")
VAL_DIR = os.path.join(DATA_DIR, "val")
TEST_DIR = os.path.join(DATA_DIR, "test")

# Kayıt Yolu ve Deney Adı
MODEL_SAVE_PATH = "../../models/pytorch" # Modelleri TF'den ayırmak için
EXPERIMENT_NAME = "resnext50_32x4d_aug5x_e30" # Deneye özel isim
os.makedirs(MODEL_SAVE_PATH, exist_ok=True) # Klasörün var olduğundan emin ol

# Eğitim parametreleri
NUM_CLASSES = 8 # Sınıf sayısı
BATCH_SIZE = 32 # VRAM'e göre ayarla (32 veya 64 ile başla)
EPOCHS = 30     
LEARNING_RATE = 0.001

# **** GPU TESPİTİ BURADA ****
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}") # Çıktı 'cuda' olmalı

In [ ]:
# ImageNet üzerinde eğitilmiş modeller için standart normalizasyon değerleri
imagenet_mean = [0.485, 0.456, 0.406]
imagenet_std = [0.229, 0.224, 0.225]

# Veri Dönüşümleri (Transforms)
data_transforms = {
    'train': transforms.Compose([
        transforms.Resize((224, 224)),
        # Offline augmentation yaptığımız için burada tekrar etmiyoruz.
        transforms.ToTensor(), 
        transforms.Normalize(imagenet_mean, imagenet_std) # Modellerin beklediği normalizasyon
    ]),
    'val': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(imagenet_mean, imagenet_std)
    ]),
    'test': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(imagenet_mean, imagenet_std)
    ]),
}

# Veri setlerini yükle
image_datasets = {
    'train': datasets.ImageFolder(TRAIN_DIR, data_transforms['train']),
    'val': datasets.ImageFolder(VAL_DIR, data_transforms['val']),
    'test': datasets.ImageFolder(TEST_DIR, data_transforms['test'])
}

# Veri yükleyicileri (Dataloader) oluştur
dataloaders = {
    x: DataLoader(image_datasets[x], batch_size=BATCH_SIZE, shuffle=(x=='train'), num_workers=2)
    for x in ['train', 'val', 'test']
}

dataset_sizes = {x: len(image_datasets[x]) for x in ['train', 'val', 'test']}
class_names = image_datasets['train'].classes

print(f"Sınıflar: {class_names}")
print(f"Eğitim verisi: {dataset_sizes['train']} görüntü") 
print(f"Validasyon verisi: {dataset_sizes['val']} görüntü") 
print(f"Test verisi: {dataset_sizes['test']} görüntü")

In [ ]:
# ResNeXt50 (32 gruplu, 4d genişlikte) modelini ImageNet ağırlıklarıyla yükle
model = models.resnext50_32x4d(weights=models.ResNeXt50_32x4d_Weights.IMAGENET1K_V2)

# 1. Transfer Öğrenme: Tüm temel katmanları dondur
for param in model.parameters():
    param.requires_grad = False

# 2. Sınıflandırıcı Kafasını Değiştir
num_ftrs = model.fc.in_features 
model.fc = nn.Linear(num_ftrs, NUM_CLASSES)

# **** MODELİ GPU'YA GÖNDERME ****
model = model.to(DEVICE)

print("Model başarıyla yüklendi ve GPU'ya taşındı.")

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.fc.parameters(), lr=LEARNING_RATE)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', patience=5, factor=0.1, verbose=True)

In [ ]:
def train_model(model, criterion, optimizer, scheduler, num_epochs=25):
    start_time = time.time()
    
    # Grafikler için geçmiş verilerini tut
    history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}

    for epoch in range(num_epochs):
        print(f'Epoch {epoch+1}/{num_epochs}')
        print('-' * 10)

        # Her epoch'ta iki faz vardır: eğitim ve validasyon
        for phase in ['train', 'val']:
            if phase == 'train':
                model.train()  # Modeli eğitim moduna al (Dropout vb. aktif)
            else:
                model.eval()   # Modeli değerlendirme moduna al (Dropout vb. kapalı)

            running_loss = 0.0
            running_corrects = 0

            # Veri üzerinde döngü
            for inputs, labels in dataloaders[phase]:
                # Verileri GPU'ya gönder
                inputs = inputs.to(DEVICE)
                labels = labels.to(DEVICE)

                # Optimizer'ın gradyanlarını sıfırla
                optimizer.zero_grad()

                # İleri besleme (forward pass)
                # Sadece 'train' fazındayken gradyanları hesapla
                with torch.set_grad_enabled(phase == 'train'):
                    outputs = model(inputs)
                    _, preds = torch.max(outputs, 1) # En yüksek skorlu sınıfı al
                    loss = criterion(outputs, labels)

                    # Sadece 'train' fazındayken geri yayılım (backward pass) + optimizasyon
                    if phase == 'train':
                        loss.backward()
                        optimizer.step()

                # İstatistikleri topla
                running_loss += loss.item() * inputs.size(0)
                running_corrects += torch.sum(preds == labels.data)

            epoch_loss = running_loss / dataset_sizes[phase]
            epoch_acc = running_corrects.double() / dataset_sizes[phase]

            print(f'{phase} Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f}')

            # Geçmişi kaydet
            if phase == 'train':
                history['train_loss'].append(epoch_loss)
                history['train_acc'].append(epoch_acc.item()) # .item() tensörü sayıya çevirir
            else:
                history['val_loss'].append(epoch_loss)
                history['val_acc'].append(epoch_acc.item())
                # LR scheduler'ı validasyon kaybına göre güncelle
                scheduler.step(epoch_loss)
        
        print() # Epoch sonu

    # Eğitim döngüsü bittikten sonra
    time_elapsed = time.time() - start_time
    print(f'Eğitim {time_elapsed // 60:.0f}dk {time_elapsed % 60:.0f}sn sürdü')

    # Modelin son halinin ağırlıklarını (state_dict) kaydet
    final_save_path = os.path.join(MODEL_SAVE_PATH, f"{EXPERIMENT_NAME}_final.pth")
    torch.save(model.state_dict(), final_save_path)
    print(f"Eğitim tamamlandı. Final model kaydedildi: {final_save_path}")
    
    return model, history

# Modeli eğitmeye başla!
model_trained, history = train_model(model, criterion, optimizer, scheduler, num_epochs=EPOCHS)

In [ ]:
train_acc = history['train_acc']
val_acc = history['val_acc']
train_loss = history['train_loss']
val_loss = history['val_loss']

plt.figure(figsize=(15, 6))

plt.subplot(1, 2, 1)
plt.plot(train_acc, label='Eğitim Doğruluğu')
plt.plot(val_acc, label='Validasyon Doğruluğu')
plt.legend()
plt.title('ResNeXt50 - Doğruluk Grafiği')

plt.subplot(1, 2, 2)
plt.plot(train_loss, label='Eğitim Kaybı')
plt.plot(val_loss, label='Validasyon Kaybı')
plt.legend()
plt.title('ResNeXt50 - Kayıp Grafiği')

plt.show()